# Parity Diffusion — Vector Field Analysis

Visualize the score function `s(x,σ) = (D(x,σ) - x)/σ²` and denoiser `D(x,σ)`
of the DiT model across training checkpoints, at critical noise scales σ ∈ [0.2, 2.0].

## 2D Slices
1. **Parity group plane** — vary 2 bits in one parity group, fix all others
2. **Two-sample plane** — interpolate between two training samples
3. **Valid↔Invalid flip axis** — vary one rule-determining bit

## Checkpoints (G=2 rep2)
- ep 1       → pre-learning (random model)
- ep 492     → just after valid onset (~50% valid at ep 436)
- ep 345511  → memorization underway (~5% mem onset at ep 323k)
- ep 999999  → final


In [ ]:
import sys
sys.path.insert(0, '/n/home12/binxuwang/Github/DiffusionAttnConsistency')

import numpy as np
import torch
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype']  = 42
matplotlib.rcParams['axes.spines.top']   = False
matplotlib.rcParams['axes.spines.right'] = False
import matplotlib.pyplot as plt
from collections import OrderedDict

from core.vector_field_lib import (
    load_model, load_training_data,
    make_group_plane_grid, make_two_sample_plane_grid,
    eval_field_on_grid, eval_score, eval_denoiser,
    project_to_axes, project_to_basis,
    plot_vector_field_2d, plot_denoiser_target_2d,
    multi_sigma_checkpoint_grid,
    parity_group_bits, valid_parity_corners,
)

SAVEROOT = '/n/holylfs06/LABS/kempner_fellow_binxuwang/Users/binxuwang/DL_Projects/DiffusionParityLearning'
FIGDIR   = '/n/home12/binxuwang/Github/DiffusionAttnConsistency/figures/vector_field'
import os; os.makedirs(FIGDIR, exist_ok=True)

DEVICE = 'cpu'   # change to 'cuda' if available
EXP_NAME = 'DiT_mini_parity_N4096_D36_G2_even_rep2'
GROUP_SIZE = 2
PARITY_VAL = 0   # even parity → valid corners: (-1,-1), (+1,+1)

## 1. Load checkpoints

In [ ]:
# Checkpoints: pre-learning / valid onset / mem onset / final
CKPT_EPOCHS = OrderedDict([
    ('ep 1\n(pre-learning)',   1),
    ('ep 492\n(valid onset)',  492),
    ('ep 345k\n(mem onset)',   345511),
    ('ep 1M\n(final)',         999999),
])

models = OrderedDict()
for label, epoch in CKPT_EPOCHS.items():
    m, sigma_data, args = load_model(EXP_NAME, epoch, device=DEVICE, saveroot=SAVEROOT)
    models[label] = m
    print(f'Loaded {label.split(chr(10))[0]}  sigma_data={sigma_data}')

# Training data
x_train = load_training_data(EXP_NAME, saveroot=SAVEROOT)   # (4096, 36)
print(f'Training data: {x_train.shape}')

## 2. Parity group plane — vary group 0 bits (b0, b1)

In [ ]:
# Pick a valid baseline sample (all groups satisfied)
x_baseline = x_train[0].numpy().copy()    # (36,) first training sample

GROUP_IDX = 0
gbits = parity_group_bits(GROUP_IDX, GROUP_SIZE)   # [0, 1]
print(f'Group {GROUP_IDX} bits: {gbits}')

# valid corners for even parity (b0*b1 = +1)
corners = valid_parity_corners(gbits, parity_val=PARITY_VAL)
print(f'Valid corners: {corners}')

# 2D grid over (b0, b1), fix all other bits to x_baseline
NGRID = 40
grid_x, v1_ax, v2_ax = make_group_plane_grid(
    x_baseline, gbits, v_range=(-2.5, 2.5), n_grid=NGRID
)
print(f'Grid shape: {grid_x.shape}')

In [ ]:
# Evaluate score magnitude heatmap + quiver for all checkpoints × σ values
SIGMAS = [0.2, 0.5, 1.0, 2.0]

fig = multi_sigma_checkpoint_grid(
    models, grid_x, SIGMAS, gbits, v1_ax, v2_ax,
    field='score_mag', device=DEVICE,
    valid_corners=corners,
    suptitle=f'{EXP_NAME}\nScore magnitude + quiver on group {GROUP_IDX} plane'
)
fig.savefig(f'{FIGDIR}/score_mag_group0_{EXP_NAME}.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Denoiser target visualization
fig = multi_sigma_checkpoint_grid(
    models, grid_x, SIGMAS, gbits, v1_ax, v2_ax,
    field='denoiser', device=DEVICE,
    valid_corners=corners,
    suptitle=f'{EXP_NAME}\nDenoiser D(x,σ) on group {GROUP_IDX} plane'
)
fig.savefig(f'{FIGDIR}/denoiser_group0_{EXP_NAME}.pdf', bbox_inches='tight')
plt.show()

## 3. Single-panel deep dive — one checkpoint, one σ, interactive

In [ ]:
# Pick one model and sigma to explore interactively
model_ep = list(models.values())[2]   # mem onset checkpoint
sigma_sel = 0.5

res = eval_field_on_grid(model_ep, grid_x, sigma_sel, device=DEVICE)

# Score components along the two group bits
u_score, v_score = project_to_axes(res['score'], gbits[0], gbits[1])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Score magnitude
plot_vector_field_2d(axes[0], v1_ax, v2_ax,
                     res['mag_score'], u_score, v_score,
                     quiver_stride=4, cmap='hot',
                     xlabel=f'bit {gbits[0]}', ylabel=f'bit {gbits[1]}',
                     title=f'Score magnitude  σ={sigma_sel}',
                     valid_corners=corners, arrow_color='white')

# Score component along b0 (does it push toward ±1?)
plot_vector_field_2d(axes[1], v1_ax, v2_ax,
                     u_score, u_score, v_score,
                     quiver_stride=4, cmap='RdBu_r',
                     clim=(-np.percentile(np.abs(u_score), 95), np.percentile(np.abs(u_score), 95)),
                     xlabel=f'bit {gbits[0]}', ylabel=f'bit {gbits[1]}',
                     title=f'Score[b{gbits[0]}]  σ={sigma_sel}',
                     valid_corners=corners)

# Denoiser pull
Du, Dv = project_to_axes(res['D'], gbits[0], gbits[1])
plot_denoiser_target_2d(axes[2], v1_ax, v2_ax, Du, Dv,
                         xlabel=f'bit {gbits[0]}', ylabel=f'bit {gbits[1]}',
                         title=f'Denoiser D(x,σ)  σ={sigma_sel}',
                         valid_corners=corners)

fig.suptitle(f'ep 345k  σ={sigma_sel}  group 0 plane', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Two-sample plane — between two training samples

In [ ]:
# Pick two training samples (memorized attractors)
x_a = x_train[0].numpy()
x_b = x_train[1].numpy()
print(f'Hamming(a,b) = {(x_a != x_b).sum()} / 36')

grid_2s, alpha_ax, beta_ax, v_ab, v_perp = make_two_sample_plane_grid(
    x_a, x_b, alpha_range=(-0.2, 1.2), beta_range=(-1.0, 1.0), n_grid=35
)
print(f'Two-sample grid: {grid_2s.shape}')

# Evaluate for final checkpoint at σ=0.5
model_final = list(models.values())[-1]
res_2s = eval_field_on_grid(model_final, grid_2s, sigma=0.5, device=DEVICE)

# Project score onto the ab direction and perpendicular
u_2s, v_2s = project_to_basis(res_2s['score'], v_ab, v_perp)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_vector_field_2d(axes[0], alpha_ax, beta_ax,
                     res_2s['mag_score'], u_2s, v_2s,
                     quiver_stride=4, cmap='viridis',
                     xlabel='α (a→b interpolation)', ylabel='β (perpendicular)',
                     title='Score magnitude  σ=0.5  final ckpt')
axes[0].axvline(0, color='lime', lw=1.5, label='sample a')
axes[0].axvline(1, color='cyan', lw=1.5, label='sample b')
axes[0].legend(fontsize=8)

Du_2s, Dv_2s = project_to_basis(res_2s['D'], v_ab, v_perp)
plot_denoiser_target_2d(axes[1], alpha_ax, beta_ax, Du_2s, Dv_2s,
                         xlabel='α (a→b)', ylabel='β (perp)',
                         title='Denoiser D(x,σ)  σ=0.5  final ckpt')
axes[1].axvline(0, color='lime', lw=1.5)
axes[1].axvline(1, color='cyan', lw=1.5)

fig.suptitle(f'Two-sample plane: train[0] ↔ train[1]', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Multiple groups — do all groups learn simultaneously?

In [ ]:
# For G=2 there are 18 groups. Compare group 0, 1, 8 (arbitrary selection)
n_groups = 36 // GROUP_SIZE
group_sel = [0, 1, n_groups//2]   # first, second, middle group

sigma_sel = 0.5
model_mid = list(models.values())[2]   # mem onset

fig, axes = plt.subplots(len(group_sel), 2, figsize=(8, 4*len(group_sel)))

for row, gidx in enumerate(group_sel):
    gb = parity_group_bits(gidx, GROUP_SIZE)
    corn = valid_parity_corners(gb, parity_val=PARITY_VAL)
    grid_g, v1, v2 = make_group_plane_grid(x_baseline, gb, v_range=(-2.5, 2.5), n_grid=35)
    res_g = eval_field_on_grid(model_mid, grid_g, sigma_sel, device=DEVICE)
    u, v = project_to_axes(res_g['score'], gb[0], gb[1])

    plot_vector_field_2d(axes[row][0], v1, v2, res_g['mag_score'], u, v,
                         quiver_stride=4, cmap='hot',
                         xlabel=f'bit {gb[0]}', ylabel=f'bit {gb[1]}',
                         title=f'Score mag  group {gidx}  σ={sigma_sel}',
                         valid_corners=corn, arrow_color='white')

    Du, Dv = project_to_axes(res_g['D'], gb[0], gb[1])
    plot_denoiser_target_2d(axes[row][1], v1, v2, Du, Dv,
                             xlabel=f'bit {gb[0]}', ylabel=f'bit {gb[1]}',
                             title=f'Denoiser  group {gidx}  σ={sigma_sel}',
                             valid_corners=corn)

fig.suptitle(f'Multiple groups comparison  σ={sigma_sel}  ep 345k', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Score magnitude along rule-violation axis (1D slice)

In [ ]:
# 1D: sweep b0 from -2.5 to 2.5, fix b1=+1 (one valid corner)
# This traces the path from valid corner (+1,+1) to invalid corner (-1,+1)

v_sweep = np.linspace(-2.5, 2.5, 80, dtype=np.float32)
x_sweep = np.tile(x_baseline, (80, 1)).astype(np.float32)
x_sweep[:, gbits[0]] = v_sweep
x_sweep[:, gbits[1]] = 1.0    # fix b1 = +1

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, (ckpt_label, model) in zip(
        [axes[0]]*len(models),  # placeholder; fill below
        models.items()):
    pass   # overwrite below

colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(models)))
SIGMAS_1d = [0.2, 0.5, 1.0, 2.0]

for col, sigma in enumerate(SIGMAS_1d[:2]):
    ax = axes[col]
    for (lbl, model), c in zip(models.items(), colors):
        score_np, _ = eval_score(model, torch.from_numpy(x_sweep), sigma, DEVICE)
        # component of score along bit b0
        s_b0 = score_np[:, gbits[0]]
        ax.plot(v_sweep, s_b0, color=c, label=lbl.replace('\n', ' '))
    ax.axvline(-1, color='gray', ls='--', lw=0.8, label='b0=-1')
    ax.axvline(+1, color='k',    ls='--', lw=0.8, label='b0=+1 (valid)')
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_xlabel(f'bit {gbits[0]} value (b1=+1 fixed)')
    ax.set_ylabel(f'score[b{gbits[0]}]')
    ax.set_title(f'1D score slice  σ={sigma}')
    ax.legend(fontsize=7)

fig.suptitle('Score along rule-violation axis: b0 sweep with b1=+1 fixed', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()